# KeetaSense - train the insect detector on Google Colab (free GPU)

This notebook fine-tunes **YOLOv8n** on the Wageningen *Yellow Sticky Traps* dataset
and produces `best.pt`, which you copy to `ml/weights/best.pt` in the project.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

Steps: 1) get the code, 2) get the dataset, 3) convert + tile, 4) train, 5) evaluate, 6) download weights.

## 1. Get the KeetaSense code
Option A: upload a zip of the project folder (without `.venv`, `.tools`, `node_modules`) using the Files panel, then unzip.
Option B: if your team pushed the project to GitHub, clone it (edit the URL).

In [ ]:
# Option A: upload keetasense.zip in the Files panel first, then run:
# !unzip -q keetasense.zip -d /content/keetasense
# Option B:
# !git clone https://github.com/<your-team>/keetasense.git /content/keetasense
%cd /content/keetasense
!ls

In [ ]:
# Colab already has a GPU build of PyTorch, so install only the other ML packages (pinned).
!pip install -q ultralytics==8.4.187 sahi==0.12.8 albumentations==2.0.8 pyyaml==6.0.3 python-dotenv==1.2.4
import torch; print('GPU available:', torch.cuda.is_available())

## 2. Get the dataset into `ml/data/raw/`
Download *Raw data from Yellow Sticky Traps with insects* (CC0, ~800 MB) from
https://data.4tu.nl/articles/_/12707066/1 . Either put `data.zip` on your Google Drive, or download it directly.

In [ ]:
# Option 1: from Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/data.zip -d ml/data/raw
# Option 2: Kaggle mirror (upload kaggle.json to the project root first)
# !pip install -q kaggle==1.7.4.5 && python -m ml.download_kaggle --dataset <owner>/<name>
!find ml/data/raw -type f | head

## 3. Convert to YOLO format (70/15/15 split) and cut training tiles

In [ ]:
!python -m ml.convert_dataset
!python -m ml.tile_dataset --tile 640 --overlap 0.2

## 4. Train
`--device 0` = first GPU. 100 epochs on a T4 takes roughly a couple of hours depending on tile count;
early stopping (`--patience 30`) stops sooner if validation stops improving.

In [ ]:
!python -m ml.train --data ml/data/yolo_tiles/data.yaml --epochs 100 --imgsz 640 --batch 32 --device 0

## 5. Evaluate on the full-size test images (YOLO + SAHI, the same pipeline the app uses)

In [ ]:
!python -m ml.evaluate --data ml/data/yolo/data.yaml --weights ml/weights/best.pt --name real_test
!cat ml/reports/eval_real_test.md

## 6. Download the weights and the report
Copy `best.pt` to `ml/weights/best.pt` on the laptop and restart the backend: the dashboard badge changes from *Demo detector* to *AI model*.

In [ ]:
from google.colab import files
files.download('ml/weights/best.pt')
files.download('ml/reports/eval_real_test.md')
files.download('ml/reports/eval_real_test.json')